# Embedding distance vs LLM judges for research-idea novelty

**Question:** on Ai2's new novelty-judge benchmark, how close do established embedding novelty metrics (no LLM at all) get to an LLM judge that uses Ai2's own prompt, and to the frontier-judge accuracies Ai2 reports?

**Setup**
- **Benchmark:** [`allenai/novelty-judge-bench`](https://huggingface.co/datasets/allenai/novelty-judge-bench) (ODC-BY). Ideas are ICLR 2026 submissions whose reviewers explicitly praised (154) or faulted (145) their originality, plus 154 ideas from a plain LLM generator. **Human-Only** is the main split; **Human+Generated** is secondary, because there the lower-novelty side is LLM-written and distance could pick up writing style.
- **Prior work:** 576k ML papers from the [Papers with Code archive](https://huggingface.co/datasets/pwc-archive/papers-with-abstracts) (CC BY-SA 4.0), which stops in July 2025, before the ICLR 2026 deadline.
- **Conditions:**
  1. **Yin:** minimum embedding distance to prior work (Yin et al., 2023)
  2. **kNN density:** mean distance to the 10 nearest papers
  3. **RND:** relative neighbour density (Wang et al., 2025)

  Conditions 1–3 each run on two embedding models: bge-small-en-v1.5 (small and fast) and BGE-M3 (what Liu & Zhai used).

  4. **LLM judge:** Qwen2.5-3B-Instruct (released before the ICLR 2026 submissions existed) on Ai2's reference prompt and two of its prompt ablations
  5. **Yin + judge:** logistic regression over both, cross-validated, once per embedding model
- **Reference points:** the six frontier judges in Ai2's Figure 4 (Claude Sonnet 4.5, Opus 4.5, Opus 4.6, GPT-5.1, GPT-5.2, GPT-5.4), on the same items and the same three prompts.
- **Metrics:** pairwise soft and strict accuracy and pointwise macro-F1 (Ai2's metrics), AUROC, seconds per 100 decisions, and a check for whether each score tracks reviewer *rating* rather than novelty.

**TL;DR:** on Ai2's Human-Only pairs, plain embedding distance with BGE-M3 (Yin's minimum distance) picks the more novel idea 60% of the time (95% CI 0.53–0.68). That is level with a 3B LLM judge on Ai2's own prompts (0.55–0.58), and 14–24 points behind Ai2's frontier judges (0.74–0.84). With the smaller bge-small model, the same metrics are no better than chance, so the embedding model matters as much as the metric. No score mostly tracks reviewer rating. On Human+Generated, the scores mostly react to whether a person or an LLM wrote the idea, not to how novel it is.

## Why this matters

Ai2's [Old Ideas, Novel Problems](https://arxiv.org/abs/2610.02022) (October 2026) showed that LLM novelty judges are unstable: rewording the prompt moves pairwise accuracy by more than 50 points, retrieval barely helps, and two purpose-built novelty systems lose to a plain prompted baseline. The study cost about $15,000 in API calls, and every condition in it is an LLM.

A separate line of work scores novelty without an LLM, as distance from prior work in embedding space. Liu & Zhai's [axiomatic benchmark](https://arxiv.org/abs/2604.15145) (2026) found the simplest of these, Yin et al.'s minimum distance, the strongest of ten systems on their probes, but it has never been run on Ai2's review-mined labels. This notebook puts the two side by side on the same items. The embedding metrics aren't new; the comparison is.

## Setup

Runs on a free Colab T4. A first run takes about 1.5–2 hours: about 10 minutes to embed the corpus with bge-small, an estimated 45–90 minutes with BGE-M3, and 15–20 minutes for the judge. Corpus embeddings and judge outputs are cached (on Drive with `USE_DRIVE`), so a rerun skips anything already done. With a complete cache from an earlier full run, a rerun takes a few minutes. No tokens or paid APIs are needed; everything is public.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

%pip install -q "transformers==5.22.1" "sentence-transformers==6.2.0" "datasets==5.1.0" "accelerate==1.15.0"

In [ ]:
import ast, gc, hashlib, json, math, re, shutil, time, urllib.request
from pathlib import Path

import jinja2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from datasets import load_dataset
from huggingface_hub import snapshot_download
from scipy.stats import binomtest, spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from importlib.metadata import version
print({p: version(p) for p in ["transformers", "sentence-transformers", "datasets", "torch", "scikit-learn", "pandas"]})
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| local")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Config

In [ ]:
SMOKE = False                     # True: 30k-paper corpus, 20 pairs and 40 pointwise ideas per split, separate cache (about 10 minutes)
USE_DRIVE = True                  # cache the corpus embeddings and judge outputs on Google Drive so a disconnect doesn't lose them

# Benchmark and corpus, pinned to the Hub revisions checked on 1 November 2026
BENCH_ID, BENCH_REV = "allenai/novelty-judge-bench", "fe201e9638efbf8562413de2d3f50f9b661910c4"
CORPUS_ID, CORPUS_REV = "pwc-archive/papers-with-abstracts", "459480aeac58509a93f5af2acd4adf20236ce8d9"
CORPUS_CUTOFF = "2025-09-19"      # ICLR 2026 abstract deadline. The archive stops in July 2025, so this is only a guard
SPLITS = {"Human-Only": "human-only", "Human+Generated": "human-plus-generated"}

# Ai2's judge prompts and criteria are read from their repo (Apache-2.0) at this commit
AI2_COMMIT = "9b2e69f3dfb46f30ffaa38414e581ad38322bf2b"

# Embedding models, both trained long before the ICLR 2026 submissions:
# name -> (Hub id, revision, max tokens, encode batch size, leakage cosine threshold)
EMBEDDERS = {
    "bge-small": ("BAAI/bge-small-en-v1.5", "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a", 256, 512, 0.95),  # 33M params, 384-d
    "bge-m3": ("BAAI/bge-m3", "5617a9f61b028005a4858fdac845db406aefb181", 512, 64, 0.95),             # 568M, 1024-d; Liu & Zhai's choice
}
EMBEDDERS_TO_RUN = ["bge-small", "bge-m3"]
# The primary model picks the retrieval judge's related work, the qualitative examples and the scatter plot.
# BGE-M3 was the stronger model in the first full run. The Yin + judge condition runs once per model.
PRIMARY = "bge-m3" if "bge-m3" in EMBEDDERS_TO_RUN else EMBEDDERS_TO_RUN[0]

# Embedding novelty metrics
KNN_K = 10                        # kNN density: mean distance to the 10 nearest papers
RND_P, RND_Q = 100, 50            # RND: P neighbours, Q papers per density estimate (the paper's values)
MARGIN = 50                       # extra neighbours fetched so P remain after the leakage filter

# LLM judges: name -> (Hub id, revision, trust_remote_code)
JUDGES = {
    "Qwen2.5-3B": ("Qwen/Qwen2.5-3B-Instruct", "aa8e72537993ba99e69dfaafa59ed015b17504d1", False),  # September 2024
}
RUN_LFM3 = False                  # True: add LFM3-1.7B (October 2026) as a possibly contaminated contrast judge
if RUN_LFM3:
    JUDGES["LFM3-1.7B"] = ("LiquidAI/LFM3-1.7B", "ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7", True)
JUDGE_DTYPE = "float16"           # T4 has no fast bfloat16. Set "float32" if the judge returns NaN probabilities (12 GB for 3B)
PROMPT_VARIANTS = ["reference", "vague_criteria", "no_criteria"]  # Ai2's reference prompt and its P1/P2 ablations
RUN_RETRIEVAL_JUDGE = False       # True: also run the reference prompt with the 5 nearest corpus abstracts as related work
RETRIEVAL_K = 5

N_SMOKE_CORPUS, N_SMOKE_ITEMS = 30_000, 20
SLUG = "2026-11-novelty-embedding-baseline"

In [ ]:
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/ML") / SLUG
else:
    CACHE = Path("outputs")
if SMOKE:
    CACHE = CACHE / "smoke"
CACHE.mkdir(parents=True, exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
OUT = Path("results"); OUT.mkdir(exist_ok=True)
print("cache:", CACHE.resolve())

## Data

### Benchmark

Each idea is an abstract. For human ideas, Ai2 had Claude Opus 4.6 strip numerical results and claims of beating specific baselines, so the text is close to, but not the same as, the submission's abstract. Pairs always share an ICLR area. Licence: **ODC-BY**.

In [ ]:
def norm_title(t):
    return re.sub(r"[^a-z0-9]", "", str(t).lower())


def text_key(t):
    return hashlib.sha1(t.strip().encode()).hexdigest()[:16]


pairs, points = {}, {}
for split, prefix in SPLITS.items():
    pw = load_dataset(BENCH_ID, f"{prefix}_pairwise", split="test", revision=BENCH_REV).to_pandas()
    pt = load_dataset(BENCH_ID, f"{prefix}_pointwise", split="test", revision=BENCH_REV).to_pandas()
    if SMOKE:
        pw = pw.sample(N_SMOKE_ITEMS, random_state=SEED)
        pt = pt.groupby("label", group_keys=False).sample(N_SMOKE_ITEMS, random_state=SEED)
    pt["y"] = (pt.label == "POSITIVE").astype(int)
    pairs[split], points[split] = pw.reset_index(drop=True), pt.reset_index(drop=True)
    print(f"{split}: {len(pw)} pairs (novel idea is B in {int(pw.expected_winner.sum())}), "
          f"{len(pt)} pointwise ideas {pt.label.value_counts().to_dict()}")

# One row per unique idea text, across both splits and formats
rows = {}
for split in SPLITS:
    for r in points[split].itertuples():
        rows.setdefault(text_key(r.idea), dict(idea=r.idea, title=r.title, forum_id=r.forum_id, source=r.idea_source,
                                               label=r.label, area=r.iclr_area, rating=r.rating))
    for r in pairs[split].itertuples():
        for s in "ab":
            t = getattr(r, f"idea_{s}")
            rows.setdefault(text_key(t), dict(idea=t, title=getattr(r, f"title_{s}"), forum_id=getattr(r, f"forum_id_{s}"),
                                              source=getattr(r, f"idea_source_{s}"), label=getattr(r, f"label_{s}"),
                                              area=r.iclr_area, rating=getattr(r, f"rating_{s}")))
ideas = pd.DataFrame.from_dict(rows, orient="index").rename_axis("key").reset_index()
key_pos = {k: n for n, k in enumerate(ideas.key)}
print(f"{len(ideas)} unique ideas:", ideas.groupby(["source", "label"]).size().to_dict())

for r in points["Human-Only"].groupby("label").head(2).head(3).itertuples():
    print(f"\n[{r.label}] {r.title} (rating {r.rating})\n{r.idea[:400]}...")

### Prior-work corpus

The [Papers with Code archive](https://huggingface.co/datasets/pwc-archive/papers-with-abstracts) (licence **CC BY-SA 4.0**) is a snapshot of 576k ML papers with arXiv abstracts and dates, frozen when the site shut down in July 2025. Liu & Zhai built their pools from the same archive. Every paper predates the ICLR 2026 deadline, which is the same restriction Ai2 applied to its retrieval. Only titles and abstracts are used; papers without a usable abstract or date, and duplicate titles, are dropped.

In [ ]:
corpus_path = CACHE / "corpus.parquet"
if corpus_path.exists():
    corpus = pd.read_parquet(corpus_path)
else:
    local = snapshot_download(CORPUS_ID, repo_type="dataset", revision=CORPUS_REV, allow_patterns=["data/*.parquet"])
    corpus = pd.read_parquet(Path(local) / "data", columns=["arxiv_id", "openreview_id", "title", "abstract", "date"])
    n0 = len(corpus)
    corpus = corpus[corpus.abstract.fillna("").str.len() > 200]
    corpus = corpus[corpus.date.notna() & (corpus.date < pd.Timestamp(CORPUS_CUTOFF))]
    corpus["tnorm"] = corpus.title.map(norm_title)
    corpus = corpus.drop_duplicates("tnorm").reset_index(drop=True)
    print(f"{n0:,} papers -> {len(corpus):,} with an abstract, a date before {CORPUS_CUTOFF} and a unique title")
    corpus.to_parquet(corpus_path)
if SMOKE and len(corpus) > N_SMOKE_CORPUS:
    corpus = corpus.sample(N_SMOKE_CORPUS, random_state=SEED).reset_index(drop=True)
print(f"corpus: {len(corpus):,} papers, {corpus.date.min():%Y-%m} to {corpus.date.max():%Y-%m}")

## Embedding novelty metrics (conditions 1–3)

Each metric is computed twice, once per embedding model:

- **bge-small-en-v1.5** (2023): 33M parameters, 384 dimensions, first 256 tokens. Small and fast.
- **BGE-M3** (2024): 568M parameters, 1024 dimensions, first 512 tokens. This is what Liu & Zhai used, so it tests whether a weak result is just down to a small embedding model.

Abstracts on both sides are embedded with the same model and compared by cosine similarity (embeddings are unit length, so a dot product). Then, for each idea:

- **Leakage filter.** Many ICLR submissions were on arXiv before the deadline, so an idea's own preprint can be in the corpus. A corpus paper is dropped from an idea's neighbours if its OpenReview ID or normalised title matches the idea, or if its cosine similarity is at least the model's leakage threshold (`EMBEDDERS`). So both models are judged against the same prior work, a paper dropped as a near-copy by an earlier model is also dropped for later ones. Every dropped neighbour is logged for a spot check.
- **Yin:** `1 − max cosine` to the remaining corpus.
- **kNN density:** `1 − mean cosine` over the `KNN_K` nearest.
- **RND** (Wang et al., 2025): the idea's neighbour density `ND` is its mean cosine distance to its `Q` nearest papers. The same is computed for each of its `P` nearest papers, using their own nearest papers. The score is the percentile of the idea's `ND` among its neighbours' (0–100): an idea in a sparser spot than its neighbours scores high.

A higher score means more novel for all three. They are deterministic, so ties are the only way a pair can be undecided.

In [ ]:
def topk_search(Qnp, k, Ct, batch=1024):
    """Top-k cosine neighbours in the corpus embeddings Ct. Returns (similarities, corpus row indices).

    The search runs in float16, but float16 similarities are only good to about 0.0005 near 0.9,
    which leaves many exact ties between ideas. So the k hits are rescored in float32 and re-sorted.
    """
    sims, idx = [], []
    for i in range(0, len(Qnp), batch):
        q = torch.from_numpy(np.asarray(Qnp[i:i + batch])).to(DEVICE, Ct.dtype)
        _, j = (q @ Ct.T).topk(k, dim=1)
        s = torch.einsum("bd,bkd->bk", q.float(), Ct[j].float())
        s, order = s.sort(dim=1, descending=True)
        sims.append(s.cpu().numpy()); idx.append(j.gather(1, order).cpu().numpy())
    return np.concatenate(sims), np.concatenate(idx)


or_map = {k: set(v) for k, v in corpus.dropna(subset=["openreview_id"]).groupby("openreview_id").groups.items()}
t_map = dict(zip(corpus.tnorm, corpus.index))
near_copies = {k: set() for k in ideas.key}      # corpus rows dropped as near-copies, shared across models
METRICS, METRIC_EMB, NEIGHBOURS, EMB_TIMING, leak_rows = {}, {}, {}, {}, []

for emb in EMBEDDERS_TO_RUN:
    emb_name, emb_rev, emb_len, emb_batch, leak_cos = EMBEDDERS[emb]
    print(f"\n=== {emb} ({emb_name})")
    embedder = SentenceTransformer(emb_name, revision=emb_rev, device=DEVICE)
    embedder.max_seq_length = emb_len
    if DEVICE == "cuda":
        embedder.half()

    def embed(texts):
        return embedder.encode(list(texts), batch_size=emb_batch, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=len(texts) > 1000).astype(np.float16)

    emb_path, timing_path = CACHE / f"corpus_emb_{emb}.npy", CACHE / f"timing_{emb}.json"
    if emb_path.exists() and len(np.load(emb_path, mmap_mode="r")) == len(corpus):
        C = np.load(emb_path)
        corpus_embed_s = json.loads(timing_path.read_text())["corpus_embed_s"] if timing_path.exists() else float("nan")
    else:
        t0 = time.time()
        C = embed(corpus.abstract)
        corpus_embed_s = time.time() - t0
        np.save(emb_path, C)
        timing_path.write_text(json.dumps({"corpus_embed_s": corpus_embed_s, "n": len(corpus)}))
    t0 = time.time()
    Q = embed(ideas.idea)
    idea_embed_s = time.time() - t0
    del embedder; gc.collect()
    print(f"corpus {C.shape}: {corpus_embed_s / 60:.1f} min to embed (one-off); {len(Q)} ideas: {idea_embed_s:.1f} s")

    Ct = torch.from_numpy(C).to(DEVICE)
    if DEVICE == "cpu":
        Ct = Ct.float()
    t0 = time.time()
    S, I = topk_search(Q, RND_P + MARGIN, Ct)

    # Leakage filter
    keep_s, keep_i = [], []
    for n, r in enumerate(ideas.itertuples()):
        banned = set(or_map.get(r.forum_id, ())) if isinstance(r.forum_id, str) else set()
        if r.source == "human" and t_map.get(norm_title(r.title)) is not None:
            banned.add(t_map[norm_title(r.title)])
        ok = []
        for s, j in zip(S[n], I[n]):
            if j in banned or j in near_copies[r.key] or s >= leak_cos:
                reason = ("id/title match" if j in banned else
                          f"cos >= {leak_cos}" if s >= leak_cos else "near-copy for an earlier model")
                if s >= leak_cos and j not in banned:
                    near_copies[r.key].add(j)
                leak_rows.append(dict(embedder=emb, key=r.key, idea_title=r.title, corpus_title=corpus.title[j],
                                      cos=round(float(s), 3), reason=reason))
            else:
                ok.append((s, j))
        if len(ok) < RND_P:
            raise RuntimeError(f"{emb}, idea {r.key}: only {len(ok)} neighbours left after the leakage filter; raise MARGIN")
        keep_s.append([s for s, _ in ok[:RND_P]]); keep_i.append([j for _, j in ok[:RND_P]])
    NS, NI = np.array(keep_s), np.array(keep_i)

    # Scores
    ideas[f"yin_{emb}"] = 1 - NS[:, 0]
    ideas[f"knn_{emb}"] = 1 - NS[:, :KNN_K].mean(1)
    nd_idea = 1 - NS[:, :RND_Q].mean(1)
    uniq = np.unique(NI)
    s_n, _ = topk_search(C[uniq], RND_Q + 1, Ct)     # column 0 is the paper itself
    nd_all = np.full(len(corpus), np.nan)
    nd_all[uniq] = 1 - s_n[:, 1:].mean(1)
    ideas[f"rnd_{emb}"] = 100 * (nd_all[NI] <= nd_idea[:, None]).mean(1)
    ideas[f"nn_title_{emb}"], ideas[f"nn_cos_{emb}"] = corpus.title.values[NI[:, 0]], NS[:, 0]
    search_s = time.time() - t0

    for m, label in [("yin", "Yin (min distance)"), ("knn", f"kNN density (k={KNN_K})"), ("rnd", "RND")]:
        METRICS[f"{m}_{emb}"] = f"{label}, {emb}"
        METRIC_EMB[f"{m}_{emb}"] = emb
    NEIGHBOURS[emb] = (NS, NI)
    EMB_TIMING[emb] = dict(corpus_embed_min=corpus_embed_s / 60, sec_per_100=(idea_embed_s + search_s) / len(ideas) * 100)
    n_leak = sum(1 for x in leak_rows if x["embedder"] == emb)
    print(f"search + filter + scores: {search_s:.1f} s; {EMB_TIMING[emb]['sec_per_100']:.2f} s per 100 ideas incl. embedding; "
          f"leakage filter dropped {n_leak} neighbours")

    Ct = None; del C; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

leaks = pd.DataFrame(leak_rows, columns=["embedder", "key", "idea_title", "corpus_title", "cos", "reason"])
print(leaks.groupby(["embedder", "reason"]).size().to_string())
ideas[["title", "source", "label"] + list(METRICS)].head()

**Spot check the leakage filter.** The rows below are the least similar papers each model dropped by cosine similarity. They should be an idea's own preprint (same paper, maybe retitled) or a near-copy. If clearly different papers show up, raise that model's threshold in `EMBEDDERS`; if obvious preprints slip through, check the nearest-neighbour similarities in the qualitative section and lower it. The two models' similarity scales differ, so the same threshold can mean different things.

In [ ]:
with pd.option_context("display.max_colwidth", 70):
    for emb in EMBEDDERS_TO_RUN:
        print(emb)
        display(leaks[(leaks.embedder == emb) & leaks.reason.str.startswith("cos")].sort_values("cos").head(10))

## LLM judge (condition 4)

The judge gets **Ai2's own prompt and novelty criteria**, rendered from their repo at a pinned commit, in three versions:

- **reference:** Ai2's reference configuration, including the pairwise "Tip" line
- **vague_criteria:** Ai2's P1 ablation, with the guardrail sentences removed from the criteria
- **no_criteria:** Ai2's P2 ablation, with no criteria at all

**Two deviations from Ai2's setup:**

- **Log-probabilities, not generated reasoning.** Ai2's judges write a `<thinking>` block and then a JSON verdict, and each decision aggregates 3 samples. Here the reply is prefilled up to `{"novelty": ` with an empty thinking block, and the verdict is read from the probabilities of the next token being `0` or `1`. That's one deterministic forward pass per prompt, and it gives a probability you can threshold, average or feed into a classifier. It also means the small judge never gets to reason, so read its numbers as a lower bound.
- **No sampling:** each pair runs once per presentation order (A then B, and B then A), and the two probabilities are averaged, so position bias cancels out in the verdict. How often the two orders disagree is reported as the **order-flip rate**.

**Contamination.** Ai2 chose judges whose training data predates the ICLR 2026 deadline. Qwen2.5-3B-Instruct (September 2024) meets that bar. LFM3-1.7B (October 2026, optional) may have seen these papers and their reviews, so it's only a contrast.

In [ ]:
AI2 = f"https://raw.githubusercontent.com/allenai/novelty-eval/{AI2_COMMIT}/src/novelty_eval/"


def fetch(path):
    with urllib.request.urlopen(AI2 + path) as f:
        return f.read().decode()


_names = {"EVALUATION_CRITERIA", "EVALUATION_CRITERIA_RETRIEVAL", "NOVELTY_PAIRWISE_TIP", "NOVELTY_RETRIEVAL_PAIRWISE_TIP"}
AI2C = {node.targets[0].id: ast.literal_eval(node.value) for node in ast.parse(fetch("judge.py")).body
        if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name) and node.targets[0].id in _names}
VAGUE = yaml.safe_load(fetch("ablation/ablations.yaml"))["shared"]["vague_novelty_criterion"]
TEMPLATES = {n: jinja2.Template(fetch(f"templates/{n}.jinja2")) for n in
             ["pairwise_novelty", "pairwise_novelty_no_criteria", "pointwise_novelty", "pointwise_novelty_no_criteria"]}
assert _names <= set(AI2C), set(AI2C)


def with_tip(criteria, tip):
    """Ai2's get_pairwise_novelty_prompt: append the tip to the novelty criterion at the same indent."""
    text = criteria["novelty"]
    first = next(l for l in text.splitlines() if l.strip())
    return {"novelty": text.rstrip() + "\n\n" + " " * (len(first) - len(first.lstrip())) + tip}


def build_prompt(fmt, variant, texts, related=None):
    retrieval = variant == "retrieval"
    crit = AI2C["EVALUATION_CRITERIA_RETRIEVAL" if retrieval else "EVALUATION_CRITERIA"]
    if variant == "vague_criteria":
        crit = {"novelty": VAGUE}
    tmpl = TEMPLATES[f"{fmt}_novelty" + ("_no_criteria" if variant == "no_criteria" else "")]
    related = related or [""] * len(texts)
    if fmt == "pointwise":
        return tmpl.render(idea={"text": texts[0], "related_work": related[0]}, evaluation_criteria=crit)
    if variant != "no_criteria":
        crit = with_tip(crit, AI2C["NOVELTY_RETRIEVAL_PAIRWISE_TIP" if retrieval else "NOVELTY_PAIRWISE_TIP"])
    return tmpl.render(idea0={"text": texts[0], "related_work": related[0]},
                       idea1={"text": texts[1], "related_work": related[1]}, evaluation_criteria=crit)


def related_for(key):
    return "\n\n".join(f"[{i + 1}] {corpus.title[j]}: {corpus.abstract[j][:1500]}"
                       for i, j in enumerate(NEIGHBOURS[PRIMARY][1][key_pos[key], :RETRIEVAL_K]))


p = points["Human-Only"]
print(build_prompt("pairwise", "reference", [p.idea[0][:300] + " ...", p.idea[1][:300] + " ..."]))

In [ ]:
PREFILL = '<thinking>\n</thinking>\n\n```json\n{"novelty": '


def load_judge(name):
    mid, rev, remote = JUDGES[name]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev, trust_remote_code=remote)
    dtype = getattr(torch, JUDGE_DTYPE) if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(mid, revision=rev, dtype=dtype, device_map=DEVICE,
                                                 trust_remote_code=remote).eval()
    digits = [tok.encode(d, add_special_tokens=False) for d in "01"]
    assert all(len(d) == 1 for d in digits), f"'0'/'1' are not single tokens for {name}: {digits}"
    return tok, model, [d[0] for d in digits]


@torch.no_grad()
def p_one(tok, model, digits, prompt):
    """Probability that the next token after the prefilled JSON key is '1' rather than '0'."""
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tok(text + PREFILL, return_tensors="pt", add_special_tokens=False).to(model.device)
    logits = model(**enc).logits[0, -1, digits].float()
    return float(torch.softmax(logits, -1)[1]), int(enc.input_ids.shape[1])


VARIANTS = PROMPT_VARIANTS + (["retrieval"] if RUN_RETRIEVAL_JUDGE else [])
point_keys = sorted({text_key(t) for split in SPLITS for t in points[split].idea})
idea_text = dict(zip(ideas.key, ideas.idea))

J = {}
for name in JUDGES:
    cache_file = CACHE / f"judge_{name}.jsonl"
    done = {}
    if cache_file.exists():
        for line in cache_file.read_text().splitlines():
            d = json.loads(line); done[d["id"]] = d
    jobs = {}
    for variant in VARIANTS:
        for split in SPLITS:
            for r in pairs[split].itertuples():
                ka, kb = text_key(r.idea_a), text_key(r.idea_b)
                jobs[f"{name}|{variant}|pair|{ka}|{kb}"] = ("pairwise", variant, [ka, kb])
                jobs[f"{name}|{variant}|pair|{kb}|{ka}"] = ("pairwise", variant, [kb, ka])
        for k in point_keys:
            jobs[f"{name}|{variant}|point|{k}"] = ("pointwise", variant, [k])
    todo = [j for j in jobs if j not in done]
    print(f"{name}: {len(jobs)} prompts, {len(todo)} still to run")
    if todo:
        tok, model, digits = load_judge(name)
        with open(cache_file, "a") as f:
            for jid in tqdm(todo, desc=name):
                fmt, variant, keys = jobs[jid]
                related = [related_for(k) for k in keys] if variant == "retrieval" else None
                t0 = time.time()
                p1, ntok = p_one(tok, model, digits, build_prompt(fmt, variant, [idea_text[k] for k in keys], related))
                if not math.isfinite(p1):
                    raise RuntimeError(f"{name} returned NaN on {jid}; set JUDGE_DTYPE = 'float32' and rerun")
                done[jid] = dict(id=jid, p1=p1, tokens=ntok, seconds=time.time() - t0)
                f.write(json.dumps(done[jid]) + "\n"); f.flush()
        del model; gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    J.update({j: done[j] for j in jobs})
print(f"{len(J)} judge decisions; mean {np.mean([d['tokens'] for d in J.values()]):.0f} prompt tokens, "
      f"{np.mean([d['seconds'] for d in J.values()]):.2f} s each")

## Evaluation

Ai2's metrics, applied to every condition in the same way:

- **Pairwise soft accuracy** (ties earn half credit) and **strict accuracy** (ties earn nothing). A score-based condition picks the idea with the higher score.
- **Pointwise macro-F1.** The embedding scores have no natural cut-off, so each condition gets a threshold chosen on the other folds of a 5-fold split (**CV F1**). The judge also gets the plain 0.5 cut-off (**F1@0.5**), which is how Ai2 used its judges.
- **Uncertainty:** each pairwise strict accuracy gets a 95% bootstrap confidence interval (5,000 resamples of the pairs) and an exact binomial p-value against chance (0.5). With 154 pairs, an accuracy needs to be about 0.58 to reach p < 0.05, and with six embedding conditions per split, a Bonferroni correction lowers the bar to p < 0.008 (about 0.61).
- **AUROC:** how well the score separates novel from lower-novelty ideas, at any threshold.
- **Yin + judge:** run once per embedding model, a logistic regression on that model's Yin score and the judge's reference-prompt probability, with 5-fold cross-validated predictions. For pairs, the features are the differences between the two ideas. The regression is free to give a score a negative weight, so if the judge is reliably *wrong* (for example, always preferring generated ideas), it learns to flip it. A high combined score next to a below-chance judge means exactly that, not that the two scores agree.
- **Reading the judge's F1@0.5:** a small judge read through log-probabilities is often skewed to one answer, so every idea lands on the same side of 0.5 and F1@0.5 sits at about 0.33. CV F1 and AUROC show whether there is signal underneath.

In [ ]:
SC = ideas.set_index("key")
CORRECT = {}    # (split, condition) -> per-pair correctness, for the paired tests below


def pair_eval(score_a, score_b, winner, key=None):
    """Soft and strict accuracy, with a 95% bootstrap CI and an exact binomial p-value (vs 0.5) for strict accuracy."""
    d = np.sign(np.asarray(score_b, float) - np.asarray(score_a, float))   # +1: B wins, -1: A wins, 0: tie
    truth = np.where(np.asarray(winner) == 1, 1, -1)
    correct, ties = d == truth, d == 0
    if key is not None:
        CORRECT[key] = correct
    boot = correct[np.random.default_rng(SEED).integers(0, len(correct), (5000, len(correct)))].mean(1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return dict(pair_soft=float((correct + 0.5 * ties).mean()), pair_strict=float(correct.mean()), pair_ties=int(ties.sum()),
                strict_ci_lo=float(lo), strict_ci_hi=float(hi),
                strict_p_vs_chance=float(binomtest(int(correct.sum()), len(correct), 0.5).pvalue))


def cv_f1(score, y, folds=5):
    score, y = np.asarray(score, float), np.asarray(y)
    pred = np.zeros_like(y)
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(score, y):
        best = max(np.unique(score[tr]), key=lambda t: f1_score(y[tr], score[tr] >= t, average="macro"))
        pred[te] = score[te] >= best
    return float(f1_score(y, pred, average="macro"))


def cv_logreg(X, y, folds=5):
    """Out-of-fold probabilities from a standardised logistic regression."""
    X, y = np.asarray(X, float), np.asarray(y)
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    out = np.zeros(len(y))
    for tr, te in StratifiedKFold(folds, shuffle=True, random_state=SEED).split(X, y):
        out[te] = LogisticRegression().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return out


def judge_pair_probs(name, variant, pw):
    """P(B is more novel), averaged over both presentation orders, and whether the two orders disagree."""
    ka, kb = pw.idea_a.map(text_key), pw.idea_b.map(text_key)
    p_ab = np.array([J[f"{name}|{variant}|pair|{a}|{b}"]["p1"] for a, b in zip(ka, kb)])       # P(second idea, B)
    p_ba = np.array([J[f"{name}|{variant}|pair|{b}|{a}"]["p1"] for a, b in zip(ka, kb)])       # P(second idea, A)
    return (p_ab + 1 - p_ba) / 2, (p_ab > 0.5) != (p_ba < 0.5)


def judge_point_probs(name, variant, pt):
    return np.array([J[f"{name}|{variant}|point|{text_key(t)}"]["p1"] for t in pt.idea])


def sec_per_100(name, variant, kind):
    s = [d["seconds"] for k, d in J.items() if k.startswith(f"{name}|{variant}|{kind}|")]
    return float(np.mean(s)) * (200 if kind == "pair" else 100)    # a pair costs two prompts


rows = []
for split in SPLITS:
    pw, pt = pairs[split], points[split]
    ka, kb, kp = pw.idea_a.map(text_key), pw.idea_b.map(text_key), pt.idea.map(text_key)
    for m, label in METRICS.items():
        sp = SC.loc[kp, m].values
        rows.append(dict(split=split, condition=label, **pair_eval(SC.loc[ka, m].values, SC.loc[kb, m].values, pw.expected_winner,
                                                                  key=(split, label)),
                         point_f1_cv=cv_f1(sp, pt.y), point_auroc=roc_auc_score(pt.y, sp),
                         sec_per_100=EMB_TIMING[METRIC_EMB[m]]["sec_per_100"]))
    for name in JUDGES:
        for variant in VARIANTS:
            pb, flips = judge_pair_probs(name, variant, pw)
            pp = judge_point_probs(name, variant, pt)
            rows.append(dict(split=split, condition=f"{name} judge ({variant})",
                             **pair_eval(1 - pb, pb, pw.expected_winner, key=(split, f"{name} judge ({variant})")),
                             order_flip=float(flips.mean()), point_f1_cv=cv_f1(pp, pt.y),
                             point_f1_05=float(f1_score(pt.y, pp >= 0.5, average="macro")), point_auroc=roc_auc_score(pt.y, pp),
                             sec_per_100=sec_per_100(name, variant, "point")))
        # Yin + judge (reference prompt), once per embedding model
        pb, _ = judge_pair_probs(name, "reference", pw)
        for emb in EMBEDDERS_TO_RUN:
            yin, label = f"yin_{emb}", f"Yin ({emb}) + {name} judge"
            Xp = np.c_[SC.loc[kb, yin].values - SC.loc[ka, yin].values, pb - 0.5]
            pc = cv_logreg(Xp, pw.expected_winner)
            pp = cv_logreg(np.c_[SC.loc[kp, yin].values, judge_point_probs(name, "reference", pt)], pt.y)
            rows.append(dict(split=split, condition=label, **pair_eval(1 - pc, pc, pw.expected_winner, key=(split, label)),
                             point_f1_cv=float(f1_score(pt.y, pp >= 0.5, average="macro")), point_auroc=roc_auc_score(pt.y, pp)))

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "summary.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    display(summary)

### Paired tests

Two conditions scored on the same 154 pairs can be compared more sharply than their two confidence intervals suggest, because most pairs are easy or hard for both. The exact McNemar test below uses only the pairs where one condition is right and the other wrong. It compares each embedding metric across the two models, and each model's Yin score with each judge prompt.

In [ ]:
def mcnemar(a, b):
    """Exact McNemar test on two conditions' per-pair correctness."""
    n_a, n_b = int((a & ~b).sum()), int((~a & b).sum())
    return n_a, n_b, (float(binomtest(n_a, n_a + n_b, 0.5).pvalue) if n_a + n_b else 1.0)


comparisons = [(METRICS[f"{m}_{e1}"], METRICS[f"{m}_{e2}"])
               for m in ("yin", "knn", "rnd") for e1, e2 in zip(EMBEDDERS_TO_RUN, EMBEDDERS_TO_RUN[1:])]
comparisons += [(METRICS[f"yin_{emb}"], f"{name} judge ({variant})")
                for name in JUDGES for variant in VARIANTS for emb in EMBEDDERS_TO_RUN]
paired = []
for split in SPLITS:
    for a, b in comparisons:
        ca, cb = CORRECT[(split, a)], CORRECT[(split, b)]
        n_a, n_b, p = mcnemar(ca, cb)
        paired.append(dict(split=split, condition_a=a, condition_b=b, strict_a=ca.mean(), strict_b=cb.mean(),
                           a_right_b_wrong=n_a, b_right_a_wrong=n_b, p=p))
paired = pd.DataFrame(paired)
paired.to_csv(OUT / "paired_tests.csv", index=False)
print(f"Bonferroni threshold for the {len(METRICS)} embedding conditions against chance: p < {0.05 / len(METRICS):.4f}")
with pd.option_context("display.float_format", "{:.3f}".format):
    display(paired)

### Ai2's frontier judges (Figure 4)

Ai2's Figure 4 reports six frontier judges on the same items. Its prompt rows P1 ("criteria without guardrails") and P2 ("criteria removed") are this notebook's `vague_criteria` and `no_criteria`, so the comparison lines up prompt by prompt. The values below were read off the figure in arXiv 2610.02022v3 (rendered from the paper's SVG); pairwise is **strict** accuracy, pointwise is macro-F1.

Ai2's judges differ from this notebook's in three ways that all favour them: they are frontier models, they reason before answering (`reasoning_effort = high`), and each verdict pools 3 samples. Their pointwise verdicts are yes/no answers, so the like-for-like pointwise number here is the judge's **F1@0.5**; CV F1 is shown too.

In [ ]:
AI2_JUDGES = ["sonnet-4-5", "opus-4-5", "opus-4-6", "gpt-5.1", "gpt-5.2", "gpt-5.4"]
AI2_FIG4 = {   # (split, metric) -> prompt variant -> one value per judge, in %
    ("Human-Only", "pair_strict"): {
        "reference": [74.0, 79.2, 82.5, 83.8, 83.8, 83.1], "vague_criteria": [64.3, 79.9, 80.5, 84.4, 82.5, 81.2],
        "no_criteria": [64.3, 78.6, 81.2, 84.4, 83.8, 82.5], "retrieval": [72.1, 80.5, 83.8, 81.8, 83.8, 84.4]},
    ("Human+Generated", "pair_strict"): {
        "reference": [39.6, 63.0, 86.4, 63.0, 76.0, 77.9], "vague_criteria": [18.2, 48.0, 80.5, 53.2, 64.3, 68.2],
        "no_criteria": [23.4, 57.1, 82.5, 58.4, 75.3, 79.9], "retrieval": [51.9, 78.6, 86.4, 63.0, 74.7, 72.7]},
    ("Human-Only", "point_f1"): {
        "reference": [60.7, 70.7, 80.5, 75.2, 64.4, 66.6], "vague_criteria": [76.4, 72.4, 73.9, 69.2, 71.6, 76.6],
        "no_criteria": [71.0, 76.7, 81.2, 74.0, 58.5, 61.4], "retrieval": [68.6, 76.4, 80.2, 72.8, 66.7, 72.6]},
    ("Human+Generated", "point_f1"): {
        "reference": [69.1, 78.4, 88.9, 75.0, 64.5, 70.5], "vague_criteria": [81.8, 70.3, 89.3, 69.3, 70.8, 77.5],
        "no_criteria": [78.0, 82.4, 92.5, 75.9, 59.6, 64.6], "retrieval": [67.6, 76.6, 89.2, 73.0, 69.7, 72.5]},
}
ai2 = pd.DataFrame([dict(split=sp, metric=me, variant=v, judge=j, value=x / 100)
                    for (sp, me), by_variant in AI2_FIG4.items() for v, xs in by_variant.items()
                    for j, x in zip(AI2_JUDGES, xs)])
ai2.to_csv(OUT / "ai2_figure4.csv", index=False)

# Side by side: this notebook's conditions and Ai2's judges on their reference prompt
ref = ai2[ai2.variant == "reference"].pivot_table(index=["split", "judge"], columns="metric", values="value").reset_index()
ref = ref.sort_values("judge", key=lambda j: j.map(AI2_JUDGES.index), kind="stable")
compare = pd.concat([
    summary.assign(source="this notebook", point_f1_binary=summary.point_f1_05)[
        ["split", "condition", "source", "pair_strict", "strict_ci_lo", "strict_ci_hi", "point_f1_binary", "point_f1_cv"]],
    ref.assign(condition="Ai2: " + ref.judge + " judge (reference)", source="Ai2 Figure 4",
               pair_strict=ref.pair_strict, point_f1_binary=ref.point_f1)[
        ["split", "condition", "source", "pair_strict", "point_f1_binary"]],
], ignore_index=True)
compare.to_csv(OUT / "comparison_with_ai2.csv", index=False)

# How far each judge moves across the three prompts (pairwise strict accuracy)
spread = []
for split in SPLITS:
    for name in JUDGES:
        v = summary[(summary.split == split) & summary.condition.isin([f"{name} judge ({p})" for p in PROMPT_VARIANTS])]
        spread.append(dict(split=split, judge=name, source="this notebook", min=v.pair_strict.min(), max=v.pair_strict.max()))
    for j in AI2_JUDGES:
        v = ai2[(ai2.split == split) & (ai2.metric == "pair_strict") & (ai2.judge == j) & ai2.variant.isin(PROMPT_VARIANTS)]
        spread.append(dict(split=split, judge=j, source="Ai2 Figure 4", min=v.value.min(), max=v.value.max()))
spread = pd.DataFrame(spread).assign(range=lambda d: d["max"] - d["min"])
spread.to_csv(OUT / "prompt_spread.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    display(compare)
    display(spread)

### Does the score track novelty, or just quality?

Ai2 drew the two pools from opposite ends of the ICLR rating distribution, and Liu & Zhai found reviewers' novelty and quality judgments entangled. So a score could separate the pools because it tracks how *good* a paper is, not how *new* it is. Within a single pool, ratings vary but every idea has the same novelty label, so a strong correlation with rating inside a pool points to quality rather than novelty. Human ideas from Human-Only only.

In [ ]:
pt = points["Human-Only"]
pt = pt[pt.idea_source == "human"]
cols = {label: SC.loc[pt.idea.map(text_key), m].values for m, label in METRICS.items()}
for name in JUDGES:
    cols[f"{name} judge (reference)"] = judge_point_probs(name, "reference", pt)
conf = []
for label, v in cols.items():
    for pool, mask in [("novel pool", pt.y == 1), ("lower-novelty pool", pt.y == 0), ("both pools", pt.y >= 0)]:
        rho, pval = spearmanr(v[mask], pt.rating[mask])
        conf.append(dict(score=label, pool=pool, spearman_rho=rho, p=pval, n=int(mask.sum())))
confound = pd.DataFrame(conf)
confound.to_csv(OUT / "quality_confound.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format):
    display(confound.pivot(index="score", columns="pool", values="spearman_rho"))

## Results

In [ ]:
def bar_colour(condition):
    if condition.startswith("Ai2"):
        return "0.65"
    if "+" in condition:
        return "tab:purple"
    return "tab:orange" if "judge" in condition else "tab:blue"


n_rows = compare.groupby("split").size().max()
fig, axes = plt.subplots(1, 2, figsize=(15, 0.3 * n_rows + 1.5))
for ax, split in zip(axes, SPLITS):
    d = compare[compare.split == split]
    y = np.arange(len(d))
    ax.barh(y, d.pair_strict, color=[bar_colour(c) for c in d.condition])
    ci = d.strict_ci_lo.notna().values                     # Ai2's rows have no CI
    ax.errorbar(d.pair_strict[ci], y[ci], xerr=[d.pair_strict[ci] - d.strict_ci_lo[ci], d.strict_ci_hi[ci] - d.pair_strict[ci]],
                fmt="none", ecolor="0.25", lw=0.8, capsize=2)
    for yi, v, x in zip(y, d.pair_strict, np.fmax(d.pair_strict, d.strict_ci_hi.fillna(0))):
        ax.text(x + 0.01, yi, f"{v:.2f}", va="center", fontsize=7)
    ax.set_yticks(y, d.condition, fontsize=8); ax.invert_yaxis()
    ax.axvline(0.5, color="grey", ls="--", lw=1)
    ax.set_xlim(0, 1.0); ax.set_xlabel("Pairwise strict accuracy (ties count as wrong)"); ax.set_title(split)
fig.suptitle("Which idea is more novel? Embedding metrics (blue), small judge (orange), combined (purple), "
             "Ai2's frontier judges (grey, Figure 4).\nError bars: 95% bootstrap CI", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "pairwise_accuracy.png", dpi=150); plt.show()

In [ ]:
pt = points["Human-Only"]
kp = pt.idea.map(text_key)
first_judge = next(iter(JUDGES))
box_cols = [f"{m}_{emb}" for emb in EMBEDDERS_TO_RUN for m in ("yin", "rnd")]
fig, axes = plt.subplots(1, len(box_cols) + 1, figsize=(4 * (len(box_cols) + 1), 4))
for ax, m in zip(axes, box_cols):
    data = [SC.loc[kp[pt.y == y], m].values for y in (1, 0)]
    ax.boxplot(data, tick_labels=["novel", "lower-novelty"])
    ax.set_title(METRICS[m].replace(", ", "\n") + " (Human-Only)", fontsize=10); ax.set_ylabel(m)
pp = judge_point_probs(first_judge, "reference", pt)
for y, c, lab in [(1, "tab:green", "novel"), (0, "tab:red", "lower-novelty")]:
    axes[-1].scatter(SC.loc[kp[pt.y == y], f"yin_{PRIMARY}"], pp[pt.y == y], s=12, alpha=0.6, c=c, label=lab)
axes[-1].set_xlabel(f"Yin score ({PRIMARY})"); axes[-1].set_ylabel(f"{first_judge} P(novel)"); axes[-1].legend(fontsize=8)
axes[-1].set_title("Do the two kinds of score agree?", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "scores_by_label.png", dpi=150); plt.show()

## Qualitative examples

Human-Only pairs where Yin (on the primary embedding model) and the judge (reference prompt) disagree, plus pairs both get wrong. For each idea: its Yin score and the nearest prior paper that sets it. If a "nearest paper" looks like the idea's own preprint, the leakage filter missed it.

In [ ]:
pw = pairs["Human-Only"]
ka, kb = pw.idea_a.map(text_key), pw.idea_b.map(text_key)
yin_b_wins = SC.loc[kb, f"yin_{PRIMARY}"].values > SC.loc[ka, f"yin_{PRIMARY}"].values
pb, _ = judge_pair_probs(first_judge, "reference", pw)
truth_b = pw.expected_winner.values == 1
cases = {"Yin right, judge wrong": (yin_b_wins == truth_b) & ((pb > 0.5) != truth_b),
         "Judge right, Yin wrong": (yin_b_wins != truth_b) & ((pb > 0.5) == truth_b),
         "Both wrong": (yin_b_wins != truth_b) & ((pb > 0.5) != truth_b)}
examples = []
for case, mask in cases.items():
    print(f"{case}: {mask.sum()} pairs")
    for i in np.flatnonzero(mask)[:2]:
        r = pw.iloc[i]
        for s in "ab":
            k = text_key(r[f"idea_{s}"])
            examples.append(dict(case=case, pair=int(r.id), side=s.upper(), label=r[f"label_{s}"], title=r[f"title_{s}"],
                                 rating=r[f"rating_{s}"], yin=SC.loc[k, f"yin_{PRIMARY}"], nearest_prior=SC.loc[k, f"nn_title_{PRIMARY}"],
                                 judge_p_this=pb[i] if s == "b" else 1 - pb[i]))
examples = pd.DataFrame(examples)
with pd.option_context("display.max_colwidth", 60, "display.float_format", "{:.3f}".format):
    display(examples)

## Interpretation

**How close do the embedding metrics get?** Level with the small judge, far from the frontier. On Human-Only pairs, BGE-M3 Yin scores 0.60 (95% CI 0.53–0.68) and the Qwen2.5-3B judge 0.55–0.58 across the three prompts. The paired tests find no significant difference under any prompt (McNemar p = 0.33–0.71), and the judge itself is not significantly above chance on any of them. Ai2's frontier judges score 0.74–0.84 on the same pairs, so a cheap distance score and a small judge both sit about halfway between chance and the frontier. On single ideas the judge does better: AUROC 0.64–0.68 against 0.57 for BGE-M3 Yin and at most 0.59 for any embedding score. Combining BGE-M3 Yin with the judge adds nothing on pairs (0.60 either way).

**Which metric wins?** With BGE-M3, Liu & Zhai's ranking holds: Yin (0.60) beats kNN density (0.58) and RND (0.55). With bge-small, Yin and kNN density are at chance (0.51, 0.49), and RND is best at 0.57 without reaching significance. BGE-M3 beats bge-small on the same pairs for Yin (28 pairs against 14, p = 0.044) and kNN density (p = 0.024). The two models also disagree about which prior paper is closest: their Yin scores correlate at only 0.64, and they pick the same nearest paper for 19% of ideas. None of the six embedding conditions survives a Bonferroni correction (p < 0.008), so "Yin with BGE-M3 is above chance" is suggestive, not shown.

**Prompt sensitivity, and its embedding equivalent.** The judge moves only 3 points across the three prompts on Human-Only (0.55–0.58) and 8 on Human+Generated, within the 1–21 point range of Ai2's judges on the same prompts. Ai2's swings of more than 50 points come from its P3–P5 prompts, which aren't run here. The bigger instability is presentation order: the judge's verdict flips when the two ideas swap places on 87% of Human-Only pairs with the reference prompt (53–61% with the others), so it is mostly answering "first or second", and only averaging both orders makes it usable. The embedding metrics have no prompt and no order, but they aren't free of configuration effects: swapping bge-small for BGE-M3 moves Yin by 9 points on Human-Only and RND by 24 on Human+Generated. The embedding model plays the role the prompt plays for a judge.

**Does any score track quality instead of novelty?** No. Within each pool, where every idea has the same novelty label, no score correlates with mean reviewer rating beyond |ρ| = 0.14, and none significantly. Across both pools, the judge correlates with rating at 0.24, but the pools were drawn from opposite ends of the rating distribution, so that is expected from any score that separates them. Each pool comes from one end (the top or bottom rating decile) of each ICLR area, so ratings within a pool span a narrow range, which limits how much this check can detect.

**Human+Generated behaves differently, and writing source explains it best.** Every pair here sets a human idea against one written by Claude Sonnet 4.5. bge-small Yin rates the generated idea as more novel in 64% of pairs and lands below chance (0.36). BGE-M3 Yin is neutral (0.54). BGE-M3 RND ranks the generated idea lower in 73% of pairs and scores 0.71, but the same metric reaches only 0.55 when both ideas are human, so it is most likely telling human-written from LLM-written ideas, not judging novelty. The 3B judge prefers the generated idea in 94% of pairs with the reference prompt, and Ai2's Sonnet 4.5 judge, the same model that wrote those ideas, prefers them too (0.40). The combined conditions' 0.94–0.95 come from the regression inverting that preference. None of this split's numbers should be read as novelty accuracy.

**When would I use a distance score?** As a cheap first pass alongside a stronger judge, not as a novelty verdict. After a one-off corpus embedding, a distance score costs 1–3 seconds per 100 ideas on a T4, against 22–40 seconds for the 3B judge and API spend for a frontier one. It gives the same answer every time, has no order or prompt effects, and comes with evidence: the nearest prior papers, which a reviewer or a frontier judge can check. That makes it useful for flagging ideas that sit very close to existing work and for supplying related work to a judge. It isn't accurate enough to rank ideas on its own (0.60 against 0.74–0.84 for frontier judges), the result depends on the embedding model, and it shouldn't be used to compare human and LLM-written ideas, where it reacts to who wrote them.

## Limitations and next steps

- **Small benchmark:** 154 pairs per split, so a 95% confidence interval on pairwise accuracy is about ±8 points and differences of a few points are within noise. The paired tests help, but with this many conditions some p < 0.05 results are expected by chance.
- **The judge is small and doesn't reason.** Reading log-probabilities with an empty thinking block understates what Qwen2.5-3B could do with reasoning, and far understates frontier judges. Compare it with Ai2's numbers as a cheap baseline, not a like-for-like rerun.
- **The corpus is one snapshot.** It ends in July 2025, two months before the deadline, and Papers with Code skews towards papers with code and benchmarks. Recent or theory-heavy prior work may be missing, which makes some ideas look more novel than they are.
- **Leakage:** the filter catches exact preprints, matching titles and near-copies, but a heavily rewritten preprint below the cosine threshold would still count as prior work against its own idea.
- **Rewritten ideas, raw corpus.** The benchmark ideas are abstracts Ai2 rewrote with Claude Opus 4.6 to strip out results, while the corpus holds original abstracts. For the 31 ideas whose original is in the corpus, the rewrite moved each text far less than the distance to its nearest other paper: median cosine to its own original was 0.99 with bge-small and 0.98 with BGE-M3, against 0.91 and 0.77 to the nearest other paper. A shift shared by all ideas cancels out in pairs, but the part that varies from idea to idea adds noise unrelated to novelty, and may be part of why distance tops out around 60%. BGE-M3 also put 8 of those 31 ideas below 0.95 from their own original, so its cosine threshold alone would have missed them; the ID and title check caught them. On Human+Generated, both sides are LLM text (Opus rewrites of real abstracts against ideas Sonnet 4.5 wrote from scratch), so that split's effect is best read as real-paper ideas against generated ones, whether through content or style.
- **The labels mix novelty and quality** (see the confound check), and the Human+Generated labels are weak by Ai2's own description.
- **Next:** the retrieval-augmented judge (`RUN_RETRIEVAL_JUDGE`), the LFM3 contamination contrast, and Ai2's plan-format ideas (`*_plan` configs).

## Environment

In [ ]:
!pip freeze | grep -iE "^(transformers|sentence-transformers|datasets|accelerate|torch|scikit-learn|pandas|numpy|jinja2|huggingface)"

In [ ]:
ideas.drop(columns=["idea"]).to_csv(OUT / "idea_scores.csv", index=False)
leaks.to_csv(OUT / "leakage_dropped.csv", index=False)
examples.to_csv(OUT / "qualitative_examples.csv", index=False)
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("novelty_results", "zip", root_dir=".", base_dir="results")
    shutil.make_archive("novelty_figures", "zip", root_dir=".", base_dir="figures")
    files.download("novelty_results.zip"); files.download("novelty_figures.zip")